In [6]:
import os
from openai import OpenAI
from tqdm import tqdm
from collections import Counter

##############################################
# OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

##############################################
# 1 Document Loading
##############################################

def load_docs(folder_path):

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                content = f.read()

                documents.append({
                    "id": file,
                    "raw": content
                })

    return documents


##############################################
# 2 LLM Reasoning (Direct)
##############################################

def llm_reasoning(query_raw):


    prompt = f"""

Identify only the design pattern Singleton used in the following PlantUML diagram. 
If no Singleton is found, respond with "nons".

QUERY:
{query_raw}



Answer ONLY with (Singleton or nons).
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()


##############################################
# 3 Pattern Detection
##############################################

def detect_pattern(test_doc):

    prediction = llm_reasoning(test_doc["raw"])

    return prediction


##############################################
# 4 Evaluation
##############################################

def evaluate_detection(test_folder):

    test_docs = load_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]

        total_counts[true_label] += 1

        predicted_label = detect_pattern(doc)

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"File: {doc['id']} | True: {true_label} | Pred: {predicted_label}")

    per_pattern_accuracy = {}

    for label in total_counts:
        per_pattern_accuracy[label] = correct_counts[label] / total_counts[label]

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")

    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

In [7]:
test_folder = "PMART_PUML/puml_test/singleton/"

overall_acc, per_pattern_acc = evaluate_detection(test_folder)

print("\nFinished evaluation.")

  1%|          | 1/93 [00:02<03:49,  2.50s/it]

File: nons_56.puml | True: nons | Pred: nons


  2%|▏         | 2/93 [00:04<03:23,  2.24s/it]

File: singleton_21.puml | True: singleton | Pred: nons


  3%|▎         | 3/93 [00:06<03:08,  2.09s/it]

...

File: nons_19.puml | True: nons | Pred: nons


 99%|█████████▉| 92/93 [03:12<00:02,  2.94s/it]

File: singleton_16.puml | True: singleton | Pred: Singleton


100%|██████████| 93/93 [03:14<00:00,  2.09s/it]

File: nons_34.puml | True: nons | Pred: nons

Per-Pattern Accuracy:
nons: 0.8971
singleton: 0.9600

Overall Accuracy: 0.9140

Finished evaluation.
